In [8]:
from scipy.stats import kde
import h5py
import astropy.io.fits as fits
import csv
import pandas as pd
import numpy as np
import tables
import pickle
import os
from astropy.table import Table
from astropy.coordinates import SkyCoord
from tqdm import tqdm
from astropy.io import ascii
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import incredible as cr
from scipy.special import erf
from scipy import stats
import scipy.optimize as opt
from scipy import stats
import scipy.optimize as opt
import emcee
import tqdm
import pickle
from astropy import table
from astropy.table import Table, join, unique
from specutils import SpectralRegion
from scipy.interpolate import BSpline, make_interp_spline, UnivariateSpline
from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
import numpy as np
import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18
from astropy.cosmology import z_at_value
cosmo = Planck18

import sys
sys.path.append("/global/homes/z/zzhang13/DESI/Projection")
from setup import *
## Functions for computing spectroscopic richness and profiles
from tools.projection_functions import *

In [13]:
LF_WEIGHTED_CATALOG = 'bgs_clus_RM_gal_matched_clean_with_geoFrac_lfweight.pickle'

with open(data_dir() + LF_WEIGHTED_CATALOG, 'rb') as handle:
    bgs_matched = pickle.load(handle)

if 'lf_weight' not in bgs_matched.colnames:
    raise KeyError(
        "The input catalog does not contain 'lf_weight'. Run "
        "make_catalogs/fit_luminosity_function_weights.py first."
    )

print(f"Using catalog: {LF_WEIGHTED_CATALOG}")
print('lf_weight percentiles:', np.nanpercentile(np.asarray(bgs_matched['lf_weight'], dtype=float), [0, 16, 50, 84, 100]))
bgs_matched.columns


<TableColumns names=('TARGETID','RA_BGS','DEC_BGS','Z_BGS','WEIGHT','flux_g_dered','flux_r_dered','flux_z_dered','flux_w1_dered','flux_w2_dered','ID','LAMBDA','Z_LAMBDA','R_LAMBDA','Z_SPEC_x','RA_x','DEC_x','MODEL_MAG_R_x','MODEL_MAGERR_R_x','RM_gal_flag','Z_SPEC_y','RA_y','DEC_y','R','P','MODEL_MAG_R_y','MODEL_MAGERR_R_y','central_flag','geoFrac')>

In [14]:
bgs_matched.columns

<TableColumns names=('TARGETID','RA_BGS','DEC_BGS','Z_BGS','WEIGHT','flux_g_dered','flux_r_dered','flux_z_dered','flux_w1_dered','flux_w2_dered','ID','LAMBDA','Z_LAMBDA','R_LAMBDA','Z_SPEC_x','RA_x','DEC_x','MODEL_MAG_R_x','MODEL_MAGERR_R_x','RM_gal_flag','Z_SPEC_y','RA_y','DEC_y','R','P','MODEL_MAG_R_y','MODEL_MAGERR_R_y','central_flag','geoFrac')>

## Assign individual spectroscopic richnessto clusters

In [15]:
fcols = ['g','r','z','w1','w2']
for col in fcols:
    #bgs_matched['flux_'+col.lower()+'_dered'] = bgs_matched['FLUX_'+col]/data['MW_TRANSMISSION_'+col]
    bgs_matched['r_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_r_dered'])
    bgs_matched['g_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_g_dered'])
    bgs_matched['gmr'] = bgs_matched['g_dered']-bgs_matched['r_dered']

In [16]:
#binGap = 1e-5
wide_bin_1 = np.linspace(-0.1,-0.005,21, endpoint=False)
small_bin = np.linspace(-0.005,0.005,21, endpoint=False)
micro_bin = np.linspace(-0.005,0.005,31)
wide_bin_2 = np.linspace(0.005,0.1,21, endpoint=False)


binBoundaries = np.hstack((wide_bin_1, small_bin))
binBoundaries = np.hstack((binBoundaries, wide_bin_2))
binCent = np.asarray([(binBoundaries[i] + binBoundaries[i+1])/2 for i in range(len(binBoundaries)-1)])
binCent_micro = np.asarray([(micro_bin[i] + micro_bin[i+1])/2 for i in range(len(micro_bin)-1)])
bin_width = np.asarray([binBoundaries[i+1]-binBoundaries[i] for i in range(len(binBoundaries[:-1]))])

## None overlapping bins
assert len(set(binCent)) == len(binCent), "Overlapping bins"
assert len(set(binBoundaries)) == len(binBoundaries), "Overlapping bins"

In [17]:
#Bin by richness
lmda_bins = [[20,22],[22,25],[25,30],[30,40],[50,200]] #upper limit must match lower limit of next bin
## Bin by redshift
z_bins = [[0.1,0.2],[0.2,0.3],[0.3,0.4]]

lmda_bin_edges = np.logspace(np.log10(20), np.log10(100),11)
lmda_bins = [[lmda_bin_edges[i], lmda_bin_edges[i+1]] for i in range(len(lmda_bin_edges)-1)]

##Create an absolute magnitude column
cosmo = Planck18
h = 0.67
M_r = bgs_matched['r_dered']-cosmo.distmod(bgs_matched['Z_BGS']).value - 5*np.log10(h)
bgs_matched['M_r'] = M_r ##comoving M with h-scaling

##Cuts
bgs_matched = bgs_matched[np.where(bgs_matched['r_dered'] < 19.5)]
#bgs_matched = bgs_matched[np.where(bgs_matched['Z_BGS']< 0.3)]
#bgs_matched = bgs_matched[np.where(bgs_matched['M_r'] < 22)]
#bgs_matched = bgs_matched[np.where(bgs_matched['LAMBDA'] > 100)]

In [18]:
# Assign spectroscopic richnesses to individual clusters and save them.
# The luminosity-function correction is cluster-level:
#     lambda_true_lfweighted = lambda_true * lf_weight(ID)
#     lambda_tot_lfweighted  = lambda_tot  * lf_weight(ID)

from scipy.stats import gaussian_kde
import matplotlib.colors as colors

# One row per redMaPPer cluster after the same cuts applied above.
rmTable = unique(bgs_matched, keys='ID')

ID_list, lambda_tot_list, lambda_true_list = calc_specRichness_individual(
    binBoundaries,
    binCent,
    bgs_matched,
)

# One LF weight per cluster. This should have no scatter at fixed cluster redshift
# except for duplicated/catalog problems.
lf_by_id = {
    row['ID']: float(row['lf_weight'])
    for row in rmTable
    if np.isfinite(float(row['lf_weight'])) and float(row['lf_weight']) > 0
}

lf_weight_list = np.asarray(
    [lf_by_id.get(cluster_id, np.nan) for cluster_id in ID_list],
    dtype=float,
)

lambda_true_arr = np.asarray(lambda_true_list, dtype=float)
lambda_tot_arr = np.asarray(lambda_tot_list, dtype=float)

lambda_true_lf = lambda_true_arr * lf_weight_list
lambda_tot_lf = lambda_tot_arr * lf_weight_list

# Build an explicit ID-keyed table so the richness assignment is robust to row ordering.
spec_richness_table = Table()
spec_richness_table['ID'] = np.asarray(ID_list)
spec_richness_table['lf_weight_cluster'] = lf_weight_list
spec_richness_table['lambda_true'] = lambda_true_arr
spec_richness_table['lambda_tot'] = lambda_tot_arr
spec_richness_table['lambda_true_lfweighted'] = lambda_true_lf
spec_richness_table['lambda_tot_lfweighted'] = lambda_tot_lf

# Cluster-level table: one row per cluster, with lambda_true/lambda_tot joined on ID.
rmTable_with_spec = join(
    rmTable,
    spec_richness_table,
    keys='ID',
    join_type='inner',
)
rm_df = rmTable_with_spec.to_pandas()

# Galaxy-level table: same rows as the cut bgs_matched table, with the cluster-level
# spectroscopic richness copied onto every galaxy row belonging to that cluster.
spec_by_id_true = dict(zip(spec_richness_table['ID'], spec_richness_table['lambda_true']))
spec_by_id_tot = dict(zip(spec_richness_table['ID'], spec_richness_table['lambda_tot']))
spec_by_id_true_lf = dict(zip(spec_richness_table['ID'], spec_richness_table['lambda_true_lfweighted']))
spec_by_id_tot_lf = dict(zip(spec_richness_table['ID'], spec_richness_table['lambda_tot_lfweighted']))
lf_by_id_saved = dict(zip(spec_richness_table['ID'], spec_richness_table['lf_weight_cluster']))

bgs_matched_with_spec = bgs_matched.copy()
bgs_matched_with_spec['lambda_true'] = np.asarray(
    [spec_by_id_true.get(cluster_id, np.nan) for cluster_id in bgs_matched_with_spec['ID']],
    dtype=float,
)
bgs_matched_with_spec['lambda_tot'] = np.asarray(
    [spec_by_id_tot.get(cluster_id, np.nan) for cluster_id in bgs_matched_with_spec['ID']],
    dtype=float,
)
bgs_matched_with_spec['lambda_true_lfweighted'] = np.asarray(
    [spec_by_id_true_lf.get(cluster_id, np.nan) for cluster_id in bgs_matched_with_spec['ID']],
    dtype=float,
)
bgs_matched_with_spec['lambda_tot_lfweighted'] = np.asarray(
    [spec_by_id_tot_lf.get(cluster_id, np.nan) for cluster_id in bgs_matched_with_spec['ID']],
    dtype=float,
)
bgs_matched_with_spec['lf_weight_cluster'] = np.asarray(
    [lf_by_id_saved.get(cluster_id, np.nan) for cluster_id in bgs_matched_with_spec['ID']],
    dtype=float,
)

# Save outputs. These use the repo data directory from setup.py.
out_dir = data_dir()
rm_cluster_output_pickle = out_dir + 'rm_clusters_with_spec_richness_lfweighted.pickle'
rm_cluster_output_fits = out_dir + 'rm_clusters_with_spec_richness_lfweighted.fits'
bgs_output_pickle = out_dir + 'bgs_clus_RM_gal_matched_with_spec_richness_lfweighted.pickle'
bgs_output_fits = out_dir + 'bgs_clus_RM_gal_matched_with_spec_richness_lfweighted.fits'

with open(rm_cluster_output_pickle, 'wb') as handle:
    pickle.dump(rmTable_with_spec, handle)
rmTable_with_spec.write(rm_cluster_output_fits, overwrite=True)

with open(bgs_output_pickle, 'wb') as handle:
    pickle.dump(bgs_matched_with_spec, handle)
bgs_matched_with_spec.write(bgs_output_fits, overwrite=True)

with np.errstate(divide='ignore', invalid='ignore'):
    boost_check = np.asarray(spec_richness_table['lambda_true_lfweighted'], dtype=float) / np.asarray(spec_richness_table['lambda_true'], dtype=float)

print(f'Clusters with spectroscopic richness: {len(rmTable_with_spec):,}')
print(f'Galaxy rows saved with cluster spectroscopic richness: {len(bgs_matched_with_spec):,}')
print('Boost / lf_weight agreement percentiles:', np.nanpercentile(boost_check / lf_weight_list, [0, 16, 50, 84, 100]))
print('Saved cluster table:')
print(' ', rm_cluster_output_pickle)
print(' ', rm_cluster_output_fits)
print('Saved bgs_matched table:')
print(' ', bgs_output_pickle)
print(' ', bgs_output_fits)


In [ ]:
# Quick sanity checks after the save cell has run.
print(rmTable_with_spec[['ID', 'LAMBDA', 'lf_weight_cluster', 'lambda_true', 'lambda_tot', 'lambda_true_lfweighted', 'lambda_tot_lfweighted']][:5])
print(bgs_matched_with_spec[['ID', 'LAMBDA', 'lf_weight_cluster', 'lambda_true', 'lambda_tot', 'lambda_true_lfweighted', 'lambda_tot_lfweighted']][:5])
